# Non-Periodic Reaction-Diffusion Simulator

This notebook runs the `ReactionDiffusionNonPeriodic` simulator — a coupled PDE system modelling two interacting species $u$ and $v$ with diffusion and nonlinear reaction kinetics, but specifically bounded with **Neumann (zero-flux) boundary conditions**.

- **State variables**: `u` and `v` (species concentrations).
- **Conditioning variables**: `beta` (reaction rate strength), `d` (ratio of diffusion coefficients).
- **Dynamics**: Finite Difference Method (FDM) in real-space combined with `scipy` RK45 time integration.
- **Boundary conditions**: Non-periodic (zero-flux / Neumann) in both spatial directions, preventing flow across boundaries.

### Why this is useful

Compared to spectral FFT-based simulators which enforce periodicity, real-space FDM boundaries allow studying the effects of boundaries and restricted domains on pattern formation (such as stripes or spots aligning with walls).


## Governing equations

The two-species reaction-diffusion system takes the form:

$$
\frac{\partial u}{\partial t} = D_u \nabla^2 u + R_u(u, v; \beta),
\qquad
\frac{\partial v}{\partial t} = D_v \nabla^2 v + R_v(u, v; \beta),
$$

where $D_v / D_u = d$ and $R_u$, $R_v$ are the nonlinear reaction terms parameterised by $\beta$.

### Boundary conditions

Neumann (zero-flux) boundary conditions on the spatial domain edges:

$$
\nabla u \cdot \mathbf{n} = 0, \qquad \nabla v \cdot \mathbf{n} = 0
$$

This means no material enters or leaves the domain, and particles colliding with the edge are effectively reflected back inwards.


In [ ]:
from IPython.display import HTML

from autosim.simulations import ReactionDiffusionNonPeriodic
from autosim.utils import plot_spatiotemporal_video

In [ ]:
sim = ReactionDiffusionNonPeriodic(
    return_timeseries=True,
    log_level="progress_bar",
    n=64,
    L=20,
    T=32.2,
    dt=0.1,
    parameters_range={
        "beta": (1.0, 2.0),
        "d": (0.05, 0.3),
    },
)

batch = sim.forward_samples_spatiotemporal(n=2, random_seed=42)

print("data shape:", batch["data"].shape, "[batch, time, x, y, channels={u, v}]")
print("constant_scalars shape:", batch["constant_scalars"].shape)
print("sampled params (beta, d):", batch["constant_scalars"])

In [ ]:
anim = plot_spatiotemporal_video(
    batch["data"],
    batch_idx=0,
    channel_names=["u", "v"],
    preserve_aspect=True,
)
HTML(anim.to_jshtml())